# 6.4 Advanced Regression

**Goal:** understand polynomial, logistic, regularized, and generalized linear models through their losses and links.

**Prerequisites:** Modules 0–4 and earlier notebooks in this module; advanced notebooks also build on earlier modules 5–8. Read the section assumptions before using a method.

**Runtime:** Python 3.10+ with NumPy, SciPy, and Matplotlib. Install in your chosen environment with `python3 -m pip install numpy scipy matplotlib`, then open in JupyterLab or VS Code with a Python kernel. Use **Restart Kernel and Run All**. Each notebook is self-contained, includes saved outputs, uses synthetic data and a fixed local random seed, and needs no downloads. Numerical simulations illustrate results; they do not prove them or validate real-world assumptions.

## Module navigation

1. [Correlation](1_Correlation.ipynb)
2. [Covariance](2_Covariance.ipynb)
3. [Regression Analysis](3_Regression-Analysis.ipynb)
4. [Advanced Regression](4_Advanced-Regression.ipynb)

## Learning route

Read each definition and assumption, work through its calculation, run the code, then attempt the practice before viewing the solutions. Distinguish descriptive association, causal inference, and predictive evaluation throughout.

In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, optimize, special, linalg
rng = np.random.default_rng(2026)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.titlesize": 12, "font.size": 10})

## 1. Polynomial Regression

A model $\hat y=\beta_0+\beta_1x+\beta_2x²+\cdots$ is nonlinear in x but linear in coefficients. Increasing degree adds flexibility and overfitting risk; high powers can be ill-conditioned and extrapolate wildly. Basis scaling or orthogonal polynomials can improve numerical stability. Select degree inside training/validation, not by minimizing test error repeatedly.

**Worked example:** fit a quadratic response with an intercept, x, and x² as design columns. This remains ordinary linear least squares in the three coefficients.

In [2]:
x=np.linspace(-2,2,80);y=1+2*x+.6*x*x+rng.normal(0,.5,len(x))
P=np.column_stack([np.ones(len(x)),x,x*x]);coef=np.linalg.lstsq(P,y,rcond=None)[0]
print("Quadratic coefficients:",coef)

Quadratic coefficients: [1.15289293 2.00977933 0.53210676]


## 2. Logistic Regression

For a binary target, $Y|x\sim Bernoulli(p)$ with logit $\log[p/(1-p)]=x^T\beta$. The inverse link is the sigmoid. Logistic regression is a linear model for log-odds, not a Gaussian model for a continuous probability observation.

Minimize Bernoulli negative log-likelihood $\sum[\log(1+e^{z_i})-y_iz_i]$, using stable `logaddexp` rather than directly exponentiating huge logits. A coefficient's exponential is a conditional odds ratio per input unit, not a risk ratio. Perfect separation can make unpenalized maximum likelihood diverge; regularization and diagnostics matter.

In [3]:
X=np.column_stack([np.ones(100),rng.normal(size=100)])
y_bin=rng.binomial(1,special.expit(X@np.array([-.3,1.2])))
def logistic_loss(b):
    z=X@b
    return np.sum(np.logaddexp(0,z)-y_bin*z)
fit=optimize.minimize(logistic_loss,np.zeros(2),jac=lambda b:X.T@(special.expit(X@b)-y_bin))
print("Logistic coefficients:",fit.x,"conditional odds ratio:",np.exp(fit.x[1]))

Logistic coefficients: [-0.06513671  1.20105257] conditional odds ratio: 3.323613408903619


## 3. Ridge Regression, Lasso Regression, and Elastic Net

With centered y and standardized X (so the intercept is handled separately), a common objective is
$$\frac1{2n}\|y-Xb\|²+\lambda[\eta\|b\|_1+\tfrac12(1-\eta)\|b\|²].$$
Ridge uses η=0 (L2 shrinkage), lasso η=1 (L1, often exact zeros), and elastic net 0<η<1 (a mixture). Penalty scaling conventions differ across libraries. Features should be scaled using training data only; the intercept is commonly unpenalized.

Ridge stabilizes correlated coefficients without selecting exact zeros. Lasso can choose one among correlated predictors unstably. Elastic net can encourage grouped retention while keeping sparsity. Choose λ and η by validation inside the training process; selected coefficients do not come with ordinary unselected-model t-test guarantees.

The code uses proximal gradient with soft thresholding, an actual L1 optimization step, rather than approximating lasso by a differentiable squared penalty.

In [4]:
Xr=rng.normal(size=(100,4));Xr=(Xr-Xr.mean(0))/Xr.std(0)
yr=Xr@np.array([2.,0.,-1.,0.])+rng.normal(0,.5,100);yr-=yr.mean()
def elastic_fit(X,y,lam,ratio,steps=3000):
    n=len(y);L=np.linalg.norm(X,2)**2/n+lam*(1-ratio)
    b=np.zeros(X.shape[1]);step=1/L
    for _ in range(steps):
        g=X.T@(X@b-y)/n+lam*(1-ratio)*b
        z=b-step*g
        next_b=np.sign(z)*np.maximum(np.abs(z)-step*lam*ratio,0)
        if np.max(np.abs(next_b-b))<1e-10:b=next_b;break
        b=next_b
    return b
for ratio in [0.,.5,1.]:print("L1 ratio:",ratio,"coefficients:",elastic_fit(Xr,yr,.2,ratio))
ridge_exact=np.linalg.solve(Xr.T@Xr/len(yr)+.2*np.eye(4),Xr.T@yr/len(yr))
assert np.allclose(elastic_fit(Xr,yr,.2,0),ridge_exact,atol=1e-6)

L1 ratio: 0.0 coefficients: [ 1.72207333 -0.09699116 -0.84614841 -0.07368843]
L1 ratio: 0.5 coefficients: [ 1.78933592 -0.01921299 -0.8464381  -0.        ]
L1 ratio: 1.0 coefficients: [ 1.86760147 -0.         -0.82770357 -0.        ]


## 4. Generalized Linear Models (GLMs)

A GLM specifies an exponential-family response distribution, linear predictor η=Xβ, and link g(μ)=η connecting the conditional mean to predictors. Examples: Gaussian with identity link, Bernoulli/binomial with logit, and Poisson with log link. A GLM is not merely “linear regression with a transformed y”; its likelihood and variance function matter.

Poisson log-link regression has $E[Y|x]=e^{x^Tβ}$ and conditional variance equal to the mean under the model. An exposure t can enter as a known offset log t. Count overdispersion, zero inflation, and dependence require checking beyond a link choice. Gaussian error assumptions are not transferred automatically to every GLM.

In [5]:
Xp=np.column_stack([np.ones(120),rng.normal(size=120)])
yp=rng.poisson(np.exp(Xp@np.array([.3,.4])))
def poisson_nll(b):
    eta=Xp@b
    return np.sum(np.exp(eta)-yp*eta+special.gammaln(yp+1))
poisson_fit=optimize.minimize(poisson_nll,np.zeros(2),jac=lambda b:Xp.T@(np.exp(Xp@b)-yp))
print("Poisson log-link coefficients:",poisson_fit.x)

Poisson log-link coefficients: [0.25277513 0.52385006]


## AI/ML application and interpretation

These models support prediction, feature selection, and interpretability. Regularization improves some variance/generalization tradeoffs but does not fix leakage, biased labels, or an invalid deployment population. Report the loss, penalty convention, feature scaling, validation design, and output meaning when comparing algorithms.

## Practice

1. Why is quadratic regression still linear in parameters?
2. Is a logistic coefficient an additive probability change?
3. Which penalties can yield exact zero coefficients?
4. Why scale predictors before penalization?
5. What three components define a GLM?

## Solutions

1. x² is a fixed design column and its coefficient enters linearly.
2. No. It is additive on log-odds; the probability effect depends on the baseline/input.
3. Lasso and elastic net with a positive L1 component can; ridge generally does not.
4. Otherwise arbitrary measurement units affect the relative penalty strength.
5. Response distribution, linear predictor, and a link connecting that predictor to the conditional mean.

**Readiness check:** identify the estimand, independent unit, assumptions, uncertainty source, and an important failure mode before using these methods.

**Next:** [Continue studying](../7-Advanced-Statistical-Concepts/1_Bayesian-Statistics.ipynb).